# 02 - Data Embedding

This notebook loads the processed_dataset files and produces embedded npy dataset.

## Install needed library

In [1]:
pip install transformers torch pandas numpy tqdm

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.4 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/11.0 MB 1.4 MB/s eta 0:00:08
   --- ------------------------------------ 1.0/11.0 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/11.0 MB 1.4 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/11.0 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.8/11.0 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 2.1/11.0 MB 1.4 MB/s eta 0:00:07
   -------- ------------------------------- 2.4/11.0 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/11.0 MB 1.3 MB/s eta 0:00:07
   ---------- ----------------------------- 2.9/11.0 MB 1.3 MB/s eta 0:00:07
   ----------- ---------------------------- 3.1/11.0 MB 1.3 MB/s eta 0:00:06
   ----------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import needed library

In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, EsmModel
from tqdm import tqdm

## Initial parameter and  load ESM-2 model

### Configuration
- Sets the file paths, selects the lightweight Meta ESM-2 (8M parameters) model, and defines a batch size of 32.

### Hardware Detection
- Dynamically detects if an NVIDIA GPU (cuda) is available to speed up the embedding process; otherwise, it falls back to the cpu.

### Model Loading
- Fetches the ESM-2 model and its Tokenizer, moves the model to the selected device, and activates .eval() mode to turn off dropout, ensuring that the generated embeddings remain identical every time.

In [12]:
csv_path = "preprocessing/processed_dataset.csv"
model_name = "facebook/esm2_t6_8M_UR50D"
batch_size = 32

# 偵測是否能用 GPU 加速
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"current device: {device}")

print("loading ESM-2 model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)
model.eval()

current device: cpu
loading ESM-2 model and Tokenizer...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True, bias=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
    

## Determine the optimal padding threshold

### ESM-2 Special Token Adjustment
- The Meta ESM-2 tokenizer automatically appends a Start-of-Sequence token (cls) at the beginning and an End-of-Sequence token (eos) at the end of every peptide sequence. To prevent the model from cutting off the end of your longest sequence, the total padding ceiling must be increased by exactly 2.

In [13]:
df = pd.read_csv(csv_path)
sequences = df['sequence'].tolist()

max_seq_len = max(len(s) for s in sequences)

max_pad_len = max_seq_len + 2
print(f"Longest sequence in dataset: {max_seq_len} , Set padding longest length: {max_pad_len}")

Longest sequence in dataset: 183 , Set padding longest length: 185


## Collects the 3D token-level embedding matrix.

### Tokenizer Settings:

- Dynamically pads every shorter sequence in the batch with pad tokens until they all reach the uniform length.

- Converts the processed text tokens directly into PyTorch Tensors so the ESM-2 model can compute them.

- Disables gradient tracking. Since only extracting features and not training the ESM-2 model, this reduces memory usage by over 50% and speeds up calculation.

- Grabs the activations from the very last Transformer layer of ESM-2. Where the 320-dimensional biological meaning of each amino acid resides.

- Immediately moves the resulting matrix out of GPU VRAM back to system RAM (.cpu()) and converts it into a standard NumPy array.

- Stacks all processed mini-batches vertically to construct one unified master array.


In [14]:
all_embeddings = []

print("Start ESM-2 Token-level Embedding...")

for i in tqdm(range(0, len(sequences), batch_size)):
    batch_seqs = sequences[i:i + batch_size]
    
    inputs = tokenizer(
        batch_seqs,
        padding='max_length',
        max_length=max_pad_len,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        token_embeddings = outputs.last_hidden_state
        
        all_embeddings.append(token_embeddings.cpu().numpy())

final_embeddings = np.vstack(all_embeddings)

Start ESM-2 Token-level Embedding...


100%|██████████| 261/261 [00:46<00:00,  5.63it/s]


## Embedded data shape

### 8346 (Data Size)
Have a total of 8,346 peptide sequences processed (the combined total of training and testing sets).

### 185 (Sequence Length)
The longest raw peptide sequence in your dataset was 183 amino acids. Adding the ESM-2 special start (<cls>) and end (<eos>) tokens brought your maximum padding boundary (max_pad_len) to exactly 185. Every shorter peptide now spans exactly 185 steps, padded with zeroes/blank tokens at the tail.

### 320 (Embedding Dimension)
Each individual amino acid position has been mapped to a continuous vector of 320 dense numerical features, capturing the evolutionary and structural context provided by the ESM-2 (8M) model.

In [18]:
print("\n--- Embedding finish ---")
print(f"Matrix Shape: {final_embeddings.shape}")
print(f"Data size: {final_embeddings.shape[0]}, Sequence length: {final_embeddings.shape[1]}, embedding demintion: {final_embeddings.shape[2]}")

print("\n--- Start Splitting & Saving ---")

is_pos = (df['label'] == 1).values
is_neg = (df['label'] == 0).values
is_train = (df['set'] == 'train').values
is_test = (df['set'] == 'test').values

pos_train_features = final_embeddings[is_pos & is_train]
pos_test_features  = final_embeddings[is_pos & is_test]
neg_train_features = final_embeddings[is_neg & is_train]
neg_test_features  = final_embeddings[is_neg & is_test]

np.save("embedding/positive_train.npy", pos_train_features)
np.save("embedding/positive_test.npy",  pos_test_features)
np.save("embedding/negative_train.npy", neg_train_features)
np.save("embedding/negative_test.npy",  neg_test_features)

print(f"Saved successed")
print(f"  - positive_train.npy : {pos_train_features.shape}")
print(f"  - positive_test.npy  : {pos_test_features.shape}")
print(f"  - negative_train.npy : {neg_train_features.shape}")
print(f"  - negative_test.npy  : {neg_test_features.shape}")


--- Embedding finish ---
Matrix Shape: (8346, 185, 320)
Data size: 8346, Sequence length: 185, embedding demintion: 320

--- Start Splitting & Saving ---
Saved successed
  - positive_train.npy : (3338, 185, 320)
  - positive_test.npy  : (835, 185, 320)
  - negative_train.npy : (3338, 185, 320)
  - negative_test.npy  : (835, 185, 320)
